# 🌾 Crop-wise Random Forest Model with Visualizations
All plots are displayed inline.

In [ ]:

import pandas as pd
import numpy as np
import os
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor

%matplotlib inline


In [ ]:

df = pd.read_csv("improved_crop_dataset.csv")
df.head()


In [ ]:

cat_cols = ["state","season","crop","soil_type","soil_texture","previous_crop"]

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

os.makedirs("models", exist_ok=True)
joblib.dump(encoders, "models/encoders.pkl")


In [ ]:

results = []

for crop in df["crop"].unique():

    print(f"\n🌾 Training for: {crop}")

    crop_df = df[df["crop"] == crop]

    X = crop_df.drop("yield_t_ha", axis=1)
    y = crop_df["yield_t_ha"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=20,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))

    print(f"R2: {r2:.4f}, MAE: {mae:.3f}, RMSE: {rmse:.3f}")

    results.append([crop, r2, mae, rmse])

    joblib.dump(model, f"models/{crop}_rf_model.pkl")

    # Scatter Plot
    plt.figure()
    plt.scatter(y_test, preds)
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(f"{crop} - Actual vs Predicted")
    plt.show()

    # Residual Plot
    residuals = y_test - preds
    plt.figure()
    plt.hist(residuals, bins=30)
    plt.title(f"{crop} - Residual Distribution")
    plt.show()

    # Feature Importance
    importance = model.feature_importances_
    features = X.columns

    plt.figure()
    plt.barh(features, importance)
    plt.title(f"{crop} - Feature Importance")
    plt.show()


In [ ]:

metrics_df = pd.DataFrame(results, columns=["Crop","R2","MAE","RMSE"])
metrics_df
